# Lab 5: Aligning Two Models That Never Met

DINOv2 learned to see from photos alone, and MPNet learned to read from sentences alone. Neither has met the other,
or seen a single photo with its caption. Ask how to connect them and most practitioners will tell you to collect
captioned photos, thousands of them, and train on those.

In this lab you connect them with none. You build a bridge that carries a photo's embedding into the space where
MPNet keeps its sentences, made from the shape of each model's embeddings alone. Then you find out what it gets
right, and what it misses.

| | |
|---|---|
| Runtime | a T4 GPU, about 20 minutes of fitting; a CPU runtime works and takes about 40 |
| Models | DINOv2 ViT-B/14 for photos and MPNet for sentences, both frozen; their embeddings come with the lab |
| Data | 20,000 COCO photos and the captions of 20,000 other photos to build bridges from; 4,000 held-out photos with their own captions to test them; 5,000 CIFAR-10 photos |

> Save your own copy first: File → Save a copy in Drive.

## What is graded

- Checkpoint 1 (20%): your four functions on changed cases from the course page, and two values from Report values.
- Checkpoint 2 (30%): what makes the record of a chosen run one you could trust.
- Checkpoint 3 (35%): what your own bridge is worth next to bridges built from captioned photos.
- Checkpoint 4 (15%): a plan to rescue a bridge that fails, with a fair test.

In [ ]:
#@title Download the lab's data and code (run this cell; there is no need to read it)
import hashlib, os, pathlib, urllib.request

CODE = os.environ.get("LAB5_BASE", "https://raw.githubusercontent.com/codey-m/cv_nlp/main/labs/data/lab5/")
DATA = os.environ.get("LAB5_DATA_BASE", "https://huggingface.co/datasets/MIT-OL-AI-D/cv-nlp-lab5/resolve/main/")
HELPERS_DIGEST = "824a271586fa661bf79c77226acfd45d108e0271b11a60f72dabe0735dae3260"
DATA_DIGESTS = {'lab5_embeddings.npz': '51b0e106b9ae3e8414ac75d22021d53e0876ba9eb4f1925d7608f65dda383e89', 'lab5_maps.npz': 'c0f54d2a60d5af0d501a104b9889a39c68f9df814080f9c26ab9d29c1582462a', 'lab5_text.json': 'cacbca92f43dc6026e13368b7f2eb68c15c084443bbb90e6fd4b03900372aae1'}

ok = os.path.exists("lab5_helpers.py") and hashlib.sha256(open("lab5_helpers.py", "rb").read()).hexdigest() == HELPERS_DIGEST
if not ok:
    if CODE.startswith("http"):
        urllib.request.urlretrieve(CODE + "lab5_helpers.py", "lab5_helpers.py")
    else:
        pathlib.Path("lab5_helpers.py").write_bytes((pathlib.Path(CODE) / "lab5_helpers.py").read_bytes())
if hashlib.sha256(open("lab5_helpers.py", "rb").read()).hexdigest() != HELPERS_DIGEST:
    raise RuntimeError("lab5_helpers.py did not download intact. Delete it and run this cell again.")
import lab5_helpers as H
for name, digest in DATA_DIGESTS.items():
    H.fetch(name, digest, DATA, "lab5_data")
data = H.Data("lab5_data")
print(f"{len(data.X):,} photos and the captions of {len(data.Y):,} other photos to build bridges from;",
      f"{len(data.VX):,} held-out photos with their own captions;", f"{len(data.CX):,} CIFAR-10 photos.",
      "Running on", "a GPU." if H.DEVICE.type == "cuda" else "the CPU.")

In [ ]:
#@title Keep your fits in Google Drive (recommended; run this cell)
USE_DRIVE = True  #@param {type:"boolean"}
folder = "lab5_runs"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        folder = "/content/drive/MyDrive/lab5_runs"
    except Exception:
        print("Drive is not mounted, so fits are kept for this session only.")
SAVED = H.Saved(folder)
print("Fits are saved in", folder, "as they finish; Run all skips any that are already there.")

In [ ]:
import math
import time
import torch

## Pick three photos

These are 12 of the 4,000 held-out photos. People wrote five or so captions for each one. MPNet's embedding of a
photo's captions averages them all; below each photo you see the first.

In [ ]:
GALLERY = [764, 4, 762, 606, 1155, 849, 818, 570, 105, 30, 35, 641]   # rows of the held-out set
H.show_photos([data.ids[i] for i in GALLERY], [str(k) for k in range(1, 13)])
for k, i in enumerate(GALLERY, 1):
    print(f"{k:2d}. {data.captions[i][0]}")

Pick three to follow through the lab by their numbers, or keep the three already chosen.

In [ ]:
WATCH = [1, 2, 3]   # your three photos, by gallery number
assert len(WATCH) == 3 and len(set(WATCH)) == 3 and all(1 <= w <= 12 for w in WATCH), "Choose three different numbers from 1 to 12."

Predict: once there is a bridge built without a single captioned photo, where will each of your photos' own captions
land among the 4,000 held-out captions? For each photo choose "first", "top 40", "top 400" or "around 2,000", which
is where a coin flip would put it.

In [ ]:
PREDICT_WATCH = [None, None, None]   # one choice per photo, in the order of WATCH

In [ ]:
CHOICES = ("first", "top 40", "top 400", "around 2,000")
assert len(PREDICT_WATCH) == 3 and all(p in CHOICES for p in PREDICT_WATCH), f"Choose each from {CHOICES}."

## A bridge that knows nothing

A bridge is a 768 × 768 matrix W. It carries a photo's embedding x to xW, and the similarity of photo i and caption j
is the inner product of x_i W with y_j. Every bridge in this lab is a different way to get W. The first is a random
rotation.

## Your task 1: Score a bridge

For each held-out photo, count the captions the bridge scores higher than the photo's own, and divide by the number
of other captions. `foscttm` averages that fraction over the photos; the name stands for "fraction of samples closer
than the true match". A perfect bridge scores 0, and one that knows nothing scores about 0.5.

<details><summary>Hint</summary>

Take one photo at a time. Its own caption is where its row meets the matching column, and every other entry in the
row is a caption competing with it. The photo's share is the fraction of competitors that beat its own caption
outright; the score averages that share over the photos. Work the first row of the 3 × 3 toy below by hand before
writing anything general.

</details>

In [ ]:
def foscttm(sim):
    """sim: [n, n], sim[i, j] = similarity of photo i and caption j, true partners on the diagonal."""
    # TODO: for each row, the fraction of other captions scored higher than the true one; return the mean as a float.
    raise NotImplementedError("Compute FOSCTTM.")

In [ ]:
toy = torch.tensor([[0.9, 0.1, 0.2], [0.8, 0.3, 0.5], [0.1, 0.2, 0.7]])
assert abs(foscttm(toy) - 1 / 3) < 1e-9, "Row 1 has two captions above its own; rows 0 and 2 have none."
assert foscttm(torch.eye(5)) == 0.0
assert abs(foscttm(-torch.eye(4)) - 1.0) < 1e-9
assert foscttm(torch.tensor([[1.0, 1.0], [0.0, 1.0]])) == 0.0, "A caption that ties with the right one does not count."
print("Task 1 checks pass.")

### Provided: the scoreboard

`score` reports a bridge's FOSCTTM, how often it puts a photo's own caption first, how often in the top 40, and how
many CIFAR-10 photos it labels correctly when each is given the class whose sentence ("a photo of a dog.") it scores
highest. Then it shows where your three photos' captions land, and the caption the bridge puts first for each.

In [ ]:
BOARD, JOURNEY = [], {}


def score(name, W):
    sim = (data.VX @ W) @ data.VY.T
    pos = (sim > sim.diag()[:, None]).sum(1) + 1
    row = {"bridge": name, "FOSCTTM": foscttm(sim), "first": (pos == 1).float().mean().item(),
           "top 40": (pos <= 40).float().mean().item(), "CIFAR-10": H.cifar_accuracy(W, data)}
    BOARD.append(row)
    JOURNEY[name] = [int(pos[GALLERY[w - 1]]) for w in WATCH]
    print(f"{name}: FOSCTTM {row['FOSCTTM']:.3f} | own caption first {row['first']:.1%} | "
          f"in the top 40 {row['top 40']:.1%} | CIFAR-10 {row['CIFAR-10']:.1%}")
    for w, p in zip(WATCH, JOURNEY[name]):
        top = int(sim[GALLERY[w - 1]].argmax())
        print(f"   photo {w:2d}: own caption at position {p:,}; puts first \"{data.captions[top][0]}\"")
    return pos


gen = torch.Generator(device=H.DEVICE).manual_seed(0)
W_random = torch.linalg.qr(torch.randn(768, 768, generator=gen, device=H.DEVICE))[0]
score("random rotation", W_random);

The right caption sits somewhere in the middle of 4,000, and the ten class sentences find the right CIFAR-10 class
about one time in ten. In the coding practice for lecture 5.2, the same two models compared directly labeled 1 of 20
CIFAR-10 photos.

## The ordinary way: captioned photos

Suppose someone hands you k photos with their captions, so that row i of `Xm` goes with row i of `Ym`. The rotation
that brings `Xm @ W` closest to `Ym` has a closed form, and every bridge in this lab ends with it.

## Your task 2: Build a bridge from captioned photos

<details><summary>Hint</summary>

The cross-covariance Xmᵀ Ym records how the matched rows line up, direction by direction. Its singular value
decomposition, Xmᵀ Ym = U Σ Vᵀ, splits it into a rotation, a stretch and another rotation. A bridge may turn but
never stretch, so it keeps the two rotations and drops the stretch: W = U Vᵀ.

</details>

In [ ]:
def procrustes(Xm, Ym):
    """Xm, Ym: [k, d], row i matched to row i. Return the orthogonal W that brings Xm @ W closest to Ym."""
    # TODO: the SVD of the cross-covariance, without its singular values.
    raise NotImplementedError("Solve orthogonal Procrustes.")

In [ ]:
g = torch.Generator().manual_seed(0)
Q = torch.linalg.qr(torch.randn(6, 6, generator=g, dtype=torch.float64))[0]
A = torch.randn(40, 6, generator=g, dtype=torch.float64)
W = procrustes(A, A @ Q)
assert torch.allclose(W, Q, atol=1e-8), "Rotating the rows by Q should give back Q."
assert torch.allclose(W @ W.T, torch.eye(6, dtype=torch.float64), atol=1e-8), "W must be orthogonal."
print("Task 2 checks pass.")

Predict: how many captioned photos does a bridge need before all three of your photos' captions are in the top 40?
Choose 10, 50, 100, 200, 2000, or "more" if you think 2,000 will not do it.

In [ ]:
PREDICT_PAIRS = None

In [ ]:
assert PREDICT_PAIRS in (10, 50, 100, 200, 2000, "more"), 'Choose 10, 50, 100, 200, 2000 or "more".'

In [ ]:
for k in (10, 50, 100, 200, 2000):
    score(f"{k:,} captioned photos", procrustes(data.PX[:k], data.PY[:k]))
    print()
enough = [k for k in (10, 50, 100, 200, 2000) if max(JOURNEY[f"{k:,} captioned photos"]) <= 40]
print(f"You predicted {PREDICT_PAIRS}. All three of your photos reached the top 40 at",
      f"{enough[0]:,} captioned photos." if enough else "none of these counts.")

These five bridges are the targets. From here on, the lab builds one with no captioned photos at all.

## Hide the pairs: three puzzles

Without pairs, nothing says which photo goes with which caption. What each cloud does have is a shape: how its
points sit relative to one another. Three small puzzles, about ten minutes, show what shape can tell you.

**Puzzle 1.** The squares are a copy of the circles, turned by an angle you do not know. Move the slider and run the
cell until the copy lines up, and watch the distance from A to D as it turns.

In [ ]:
#@title Puzzle 1: turn the copy back
ANGLE = 0  #@param {type:"slider", min:0, max:360, step:5}
H.puzzle_turn(ANGLE)

**Puzzle 2.** Now the copy is turned and its labels are shuffled to numbers. Which number is A? Write your matching as
the numbers that A to F go to, then run the cell to check it.

In [ ]:
H.puzzle_hidden()
GUESS = [1, 2, 3, 4, 5, 6]   # the numbers that A, B, C, D, E and F go to
H.check_guess(GUESS)

The check scores a matching by how well the two tables of relations agree: the relations among A to F against
those among the numbers they are matched to. Let the computer try all 720 matchings.

In [ ]:
H.search_matchings()

**Puzzle 3.** One more shape: four corners of a square, matched to a turned copy of themselves.

In [ ]:
H.puzzle_square()

When a shape is symmetric, geometry has nothing to choose with. Real embedding clouds come close to this. When we
checked 12 single clusterings of these embeddings, the best-agreeing matching was never the one true pairs point to,
which is why the method below averages 30 clusterings instead of trusting one.

## Line up the main directions

The first idea without pairs. Each cloud has directions along which it spreads most, its principal axes. If the two
clouds share a shape, the first axis of one should be the first axis of the other, the second the second, and so on.
An axis has no preferred sign, so `principal_axes` also flips each one to lean the same way in both spaces.

Predict: will lining up the axes move your photos' captions up from where the random rotation left them? Answer True
or False.

In [ ]:
PREDICT_AXES = None

In [ ]:
assert isinstance(PREDICT_AXES, bool), "Answer True or False."


def principal_axes(Z):
    axes = torch.linalg.svd(Z, full_matrices=False)[2]           # rows: directions of most spread, in order
    lean = torch.sign(((Z @ axes.T) ** 3).mean(0))               # make each axis lean the same way
    return axes * lean[:, None]


W_axes = principal_axes(data.X).T @ principal_axes(data.Y)      # axis k of photos to axis k of captions
score("principal axes", W_axes)
moved = sum(a < b for a, b in zip(JOURNEY["principal axes"], JOURNEY["random rotation"]))
print(f"You predicted {PREDICT_AXES}. {moved} of your 3 photos' captions moved up.")

Look at the score, then at what it puts first for your photos. Single directions are a fragile thing to match, and
groups of points are sturdier.

## Match clusters by their relations

Puzzle 2 at full size. Cluster the photos into 30 groups and, separately, the captions into 30 groups, then pair the
groups so that their relations agree. There are 30! ways to pair 30 clusters, so `H.match_clusters` searches: it
starts from 2,000 random pairings and keeps making the single swap that raises the agreement most.

## Your task 3: Score a cluster matching

`H.centered_kernel` turns a set of centers into the table of their inner products, with row and column means
removed. `cluster_agreement` takes photo centers `A`, caption centers `B` and a matching `p`, in which photo cluster
i goes with caption cluster `p[i]`, and returns how well the two tables agree under that matching. For a matching p,
the agreement is Σᵢ Σₖ K_A(i, k) · K_B(p(i), p(k)), where i and k run over all the clusters. It is the score the
puzzles used.

<details><summary>Hint</summary>

A good matching sends clusters that sit close together among the photos to clusters that sit close together among
the captions, and clusters far apart to clusters far apart. Read the caption table in the order the matching gives
and set it beside the photo table. The agreement rewards every place where the two tables are large together or
small together, and adds those rewards up over all pairs of clusters.

</details>

In [ ]:
def cluster_agreement(A, B, p):
    """A, B: [C, d] cluster centers. p: [C] long, caption cluster p[i] goes with photo cluster i. Returns a float."""
    # TODO: compare the centered kernels of A and B under the matching p.
    raise NotImplementedError("Compute the agreement.")

In [ ]:
g = torch.Generator().manual_seed(1)
A = torch.randn(8, 5, generator=g, dtype=torch.float64)
shuffle = torch.randperm(8, generator=g)
B = (A @ torch.linalg.qr(torch.randn(5, 5, generator=g, dtype=torch.float64))[0])[shuffle]   # turned, then shuffled
true_p = torch.argsort(shuffle)
best = cluster_agreement(A, B, true_p)
assert abs(best - (H.centered_kernel(A) ** 2).sum().item()) < 1e-8, "A rotation leaves every inner product unchanged."
assert all(cluster_agreement(A, B, torch.randperm(8, generator=g)) < best for _ in range(20))
print("Task 3 checks pass.")

### Provided: the cluster bridge

One clustering is a coarse and somewhat arbitrary summary, so `cluster_map` repeats the whole thing 30 times on fresh
samples and averages. Its result is a rough bridge built from 30 × 30 matched centers and nothing else. It takes about
two minutes on a T4. First, one clustering on its own:

In [ ]:
gen = torch.Generator(device=H.DEVICE).manual_seed(0)
A, B = H.kmeans(data.X[:10000], 30, gen), H.kmeans(data.Y[:10000], 30, gen)
p = H.match_clusters(A, B, gen)
p_random = torch.randperm(30, generator=gen, device=H.DEVICE)
print(f"agreement of the matching found: {cluster_agreement(A, B, p):.2f}; of a random matching: "
      f"{cluster_agreement(A, B, p_random):.2f}")


def cluster_map(Xs, Ys, gen, C=30, S=30, batch=10000):
    """The rough bridge: matched cluster centers, A^T B[p] / C, averaged over S clusterings."""
    total, batch = 0, min(batch, len(Xs), len(Ys))
    for s in range(S):
        A = H.kmeans(Xs[torch.randperm(len(Xs), generator=gen, device=H.DEVICE)[:batch]], C, gen)
        B = H.kmeans(Ys[torch.randperm(len(Ys), generator=gen, device=H.DEVICE)[:batch]], C, gen)
        total = total + A.T @ B[H.match_clusters(A, B, gen)] / C
        print(f"\r   clustering {s + 1} of {S}", end="")
    print()
    return total / S


def start(Xs, Ys, seed, name):
    """A cluster bridge, saved with the random state that comes after it, so a fit can continue from it."""
    stamp = H.stamp(cluster_map, seed=seed, rows=(len(Xs), len(Ys)), data=DATA_DIGESTS, helpers=HELPERS_DIGEST)
    saved = SAVED.get(name, stamp)
    if saved is not None:
        return saved
    gen = torch.Generator(device=H.DEVICE).manual_seed(seed)
    M = cluster_map(Xs, Ys, gen)
    return SAVED.put(name, {"M": M, "state": gen.get_state()}, stamp)


START = start(data.X, data.Y, 1, "clusters_seed1")
score("matched clusters", START["M"]);

## Refine

The cluster bridge knows only centers. The last step sharpens it with the photos and captions themselves, by
alternating two moves:

1. Match. Under the current bridge, pair each photo in a batch with one caption in a batch, one to one, so that the
   total similarity is as high as possible. `H.hungarian` solves this exactly: give it a table of scores, photos by
   captions, and it returns the indices of the matched rows and columns.
2. Refit. Treat those pairs as if they were true and run your `procrustes` on them.

The pairs are wrong at first. If they are right more often than chance, each new bridge is a little better and finds
better pairs in turn.

## Your task 4: One round of refinement

<details><summary>Hint</summary>

This is the two moves above, done once. The matching needs a score for every photo and every caption in the batch
under the current bridge. The refit needs the chosen pairs lined up row for row, each photo beside the caption it
was matched to.

</details>

In [ ]:
def refine_step(Xb, Yb, W):
    """Match the rows of Xb @ W to the rows of Yb one to one, then refit W by Procrustes on the matched rows."""
    # TODO: match the batch one to one under W, then refit on the matched pairs.
    raise NotImplementedError("Write one round of refinement.")

In [ ]:
g = torch.Generator().manual_seed(2)
Q = torch.linalg.qr(torch.randn(5, 5, generator=g, dtype=torch.float64))[0]
A = torch.randn(30, 5, generator=g, dtype=torch.float64)
B = (A @ Q)[torch.randperm(30, generator=g)]                     # the right captions, in the wrong order
rough = procrustes(A[:8], (A @ Q)[:8] + 0.3 * torch.randn(8, 5, generator=g, dtype=torch.float64))
assert not torch.allclose(rough, Q, atol=1e-3)
assert torch.allclose(refine_step(A, B, rough), Q, atol=1e-8), "One round from a rough start should recover Q here."
print("Task 4 checks pass.")

## Changed cases

Before the long runs, check your four functions on cases they have not seen. Replace the values at the top of the
next cell with the ones the course page issues for checkpoint 1, run it, and enter C1 to C4 as it prints them.

In [ ]:
# Replace these values with the inputs from the course page, then run the cell.
C1_SIM = [[0.2, 0.9, 0.4], [0.1, 0.5, 0.3], [0.6, 0.2, 0.8]]
C2_X = [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]
C2_Y = [[0.0, 1.0], [-1.0, 0.0], [-1.0, 1.0]]
C3_A = [[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0], [0.0, -2.0]]
C3_B = [[0.0, 2.0], [1.0, 0.0], [0.0, -1.0], [-1.0, 0.0]]
C3_P = [1, 2, 3, 0]
C4_X = [[1.0, 0.2], [0.1, 1.0], [-1.0, 0.3]]
C4_Y = [[-0.9, 0.1], [0.3, 1.0], [0.2, -1.0]]
C4_W = [[1.0, 0.0], [0.0, 1.0]]

t64 = lambda v: torch.tensor(v, dtype=torch.float64)
print("C1 =", round(foscttm(t64(C1_SIM)), 6))
print("C2 =", round(procrustes(t64(C2_X), t64(C2_Y))[0, 1].item(), 6), "(row 0, column 1 of W)")
print("C3 =", round(cluster_agreement(t64(C3_A), t64(C3_B), torch.tensor(C3_P)), 6))
print("C4 =", round(refine_step(t64(C4_X), t64(C4_Y), t64(C4_W))[0, 0].item(), 6), "(row 0, column 0 of the new W)")

## Report values

Run the next cell and enter D1 and D2 in checkpoint 1. Both use bridges the course built from captioned photos, so
every learner gets the same values.

In [ ]:
pos200 = H.positions(data.shipped["pairs_200"], data.VX, data.VY)
D1 = int((pos200 == 1).sum())
D2 = round(foscttm((data.VX[:1000] @ data.shipped["pairs_100"]) @ data.VY[:1000].T), 4)
print("D1 (held-out photos whose own caption the course's 200-photo bridge puts first) =", D1)
print("D2 (your foscttm for the course's 100-photo bridge on the first 1,000 held-out pairs) =", D2)

### Provided: the whole recipe

`fit` takes a cluster bridge, makes a first set of pairs from it, and then calls your `refine_step` 15 times on fresh
batches of 4,000 photos and 4,000 captions. You run it three times, with seeds 1, 2 and 3; seed 1 continues from the
cluster bridge you built above. Each run takes about three minutes on a T4.

The runs will differ, and you may not use the held-out pairs to choose among them, since then the bridge would no
longer be built without pairs. `H.matching_score` uses only the unpaired photos and captions: it matches a batch one
to one under the final bridge and reports how similar the matched pairs are. A bridge that lines the clouds up well
finds close partners for everything.

Predict: which of the bridges built from captioned photos will yours land closest to on FOSCTTM? Choose 10, 50, 100, 200 or 2000
captioned photos.

In [ ]:
PREDICT_WORTH = None

In [ ]:
assert PREDICT_WORTH in (10, 50, 100, 200, 2000), "Choose 10, 50, 100, 200 or 2000."


def fit(Xs, Ys, seed, name, rounds=15, batch=4000, begin=None):
    stamp = H.stamp(procrustes, refine_step, cluster_map, fit, seed=seed, rounds=rounds, batch=batch,
                    rows=(len(Xs), len(Ys)), begin=None if begin is None else begin["stamp"],
                    data=DATA_DIGESTS, helpers=HELPERS_DIGEST)
    saved = SAVED.get(name, stamp)
    if saved is not None:
        return saved
    t0 = time.time()
    gen = torch.Generator(device=H.DEVICE).manual_seed(seed)
    if begin is None:
        M = cluster_map(Xs, Ys, gen)
    else:
        M = begin["M"]
        gen.set_state(begin["state"].cpu())
    batch = min(batch, len(Xs), len(Ys))
    xb = Xs[torch.randperm(len(Xs), generator=gen, device=H.DEVICE)[:batch]]
    yb = Ys[torch.randperm(len(Ys), generator=gen, device=H.DEVICE)[:batch]]
    rows, columns = H.hungarian((xb @ M) @ yb.T)                # first pairs, from the cluster bridge
    W = procrustes(xb[rows], yb[columns])
    for r in range(rounds):
        xb = Xs[torch.randperm(len(Xs), generator=gen, device=H.DEVICE)[:batch]]
        yb = Ys[torch.randperm(len(Ys), generator=gen, device=H.DEVICE)[:batch]]
        W = refine_step(xb, yb, W)
        print(f"\r   {name}: round {r + 1} of {rounds}, {time.time() - t0:.0f} s", end="")
    print()
    return SAVED.put(name, {"W": W, "M": M, "matching": H.matching_score(Xs, Ys, W, batch),
                            "seconds": time.time() - t0}, stamp)


runs = {1: fit(data.X, data.Y, 1, "unpaired_seed1", begin=START),
        2: fit(data.X, data.Y, 2, "unpaired_seed2"),
        3: fit(data.X, data.Y, 3, "unpaired_seed3")}
KEPT = max(runs, key=lambda s: runs[s]["matching"])             # chosen without a single captioned photo
W_MINE = runs[KEPT]["W"]
print("Matching scores: " + ", ".join(f"seed {s} {runs[s]['matching']:.5f}" for s in runs) + f". Kept seed {KEPT}.")

The reveal: your bridge, built from no captioned photos.

In [ ]:
score("your bridge, no captioned photos", W_MINE)
H.show_photos([data.ids[GALLERY[w - 1]] for w in WATCH],
              [f"photo {w}: position {p:,}" for w, p in zip(WATCH, JOURNEY["your bridge, no captioned photos"])], 3)
print()
print(f"{'':36s}" + "".join(f"{'photo ' + str(w):>12s}" for w in WATCH))
for name, row in JOURNEY.items():
    print(f"{name:36s}" + "".join(f"{p:>12,}" for p in row))
print(f"{'you predicted':36s}" + "".join(f"{p:>12s}" for p in PREDICT_WATCH))
mine = BOARD[-1]["FOSCTTM"]
closest = min((10, 50, 100, 200, 2000), key=lambda k: abs(next(r["FOSCTTM"] for r in BOARD
                                                                if r["bridge"] == f"{k:,} captioned photos") - mine))
print(f"\nOn FOSCTTM your bridge lands closest to the {closest:,}-photo bridge; you predicted {PREDICT_WORTH:,}.")

Pause before going on: what changed your view?

## The same recipe, run again

Your three runs used the same recipe on the same data, with different random clusterings and batches.

Predict: will the run with the highest matching score also be the best of the three on the held-out pairs? Answer
True or False.

In [ ]:
PREDICT_KEPT_IS_BEST = None

In [ ]:
assert isinstance(PREDICT_KEPT_IS_BEST, bool), "Answer True or False."
pos_runs = {s: H.positions(runs[s]["W"], data.VX, data.VY) for s in runs}
fos = {s: foscttm((data.VX @ runs[s]["W"]) @ data.VY.T) for s in runs}
print(f"{'':16s}" + "".join(f"{'photo ' + str(w):>10s}" for w in WATCH) + f"{'FOSCTTM':>10s}{'matching score':>16s}")
for s in runs:
    label = f"seed {s}" + (" (kept)" if s == KEPT else "")
    print(f"{label:16s}" + "".join(f"{int(pos_runs[s][GALLERY[w - 1]]):>10,}" for w in WATCH)
          + f"{fos[s]:>10.4f}{runs[s]['matching']:>16.5f}")
stacked = torch.stack(list(pos_runs.values()))
first = stacked.min(0).values == 1
print(f"\nOf the {int(first.sum()):,} held-out photos that one of your runs puts first, "
      f"{int((first & (stacked.max(0).values > 10)).sum()):,} land outside the top 10 in another.")
best = min(fos, key=fos.get)
print(f"The matching score kept seed {KEPT}; on the held-out pairs the best run is seed {best}. "
      f"You predicted {PREDICT_KEPT_IS_BEST}.")

The matching score chose without a single captioned photo. Checking that choice against the held-out pairs afterwards
is fine; using them to make it would turn the method into a paired one. Checkpoint 2 asks what makes the record of
such a choice one you could trust.

## What crossed

A score summarizes 4,000 photos at once. Now read some of them. The next cell shows eight photos, your three and
five more, each beside the photo whose captions your bridge put first, with every caption of both.

In [ ]:
g = torch.Generator().manual_seed(5)
AUDIT = [GALLERY[w - 1] for w in WATCH] + [i for i in torch.randperm(4000, generator=g).tolist() if i not in GALLERY][:5]
SIM_MINE = (data.VX @ W_MINE) @ data.VY.T
POS_MINE = (SIM_MINE > SIM_MINE.diag()[:, None]).sum(1) + 1
for k, i in enumerate(AUDIT, 1):
    H.show_retrieval(k, i, int(SIM_MINE[i].argmax()), data, int(POS_MINE[i]))
    print()

Label each retrieval, in order, and write the few words that decided it.

| Label | What it means |
|---|---|
| own | its own captions came first |
| fits | another photo's captions that describe this one well |
| detail | the right kind of scene with a wrong detail |
| scene | the wrong scene |

In [ ]:
LABELS = [("", "")] * 8   # (label, the words that decided it), one per retrieval in the order shown

In [ ]:
assert len(LABELS) == 8 and all(label in ("own", "fits", "detail", "scene") and words.strip()
                                for label, words in LABELS), "Label all eight, each with the words that decided it."
tally = {label: sum(l == label for l, _ in LABELS) for label in ("own", "fits", "detail", "scene")}
print("Your labels:", ", ".join(f"{n} {label}" for label, n in tally.items()))
print(f"Over all 4,000 held-out photos your bridge puts the photo's own caption first for {(POS_MINE == 1).float().mean():.1%}",
      f"of them, and in the top 40 for {(POS_MINE <= 40).float().mean():.1%}.")

Two questions to answer for yourself before going on. Would this bridge help someone search a photo collection?
When it puts another photo's captions first, do they still describe this scene?

## Your bridge against captioned photos

The next cell is for checkpoint 3. It compares your kept bridge with a bridge the course built from `CHECK_PAIRS`
captioned photos on four things: overall ranking (FOSCTTM), how often each puts a photo's own caption first (depth
1), how often in the top 40 (depth 40, a first page of search results), and CIFAR-10. Each difference comes with its
paired 95% interval: the per-photo differences between the two bridges on the same photos, and how far their mean
would move with other photos like these. Run the cell as it is, then with the value checkpoint 3 issues.

In [ ]:
# Replace this value with the one from the course page for checkpoint 3, then run the cell.
CHECK_PAIRS = 50

assert CHECK_PAIRS in (50, 100, 200), "Use 50, 100 or 200."
REF = data.shipped[f"pairs_{CHECK_PAIRS}"]
POS_REF = H.positions(REF, data.VX, data.VY)
n = len(POS_MINE) - 1
print(f"Your bridge against the course's bridge from {CHECK_PAIRS} captioned photos\n")
for name, a, b, pct in [("overall ranking, FOSCTTM (lower is better)", (POS_MINE - 1).double() / n, (POS_REF - 1).double() / n, False),
                        ("own caption first (depth 1)", (POS_MINE == 1).double(), (POS_REF == 1).double(), True),
                        ("own caption in the top 40 (depth 40)", (POS_MINE <= 40).double(), (POS_REF <= 40).double(), True),
                        ("CIFAR-10", H.cifar_correct(W_MINE, data).double(), H.cifar_correct(REF, data).double(), True)]:
    m, lo, hi = H.paired_interval(a, b)
    f = (lambda v: f"{100 * v:.2f}%") if pct else (lambda v: f"{v:.4f}")
    d = (lambda v: f"{100 * v:+.2f} points") if pct else (lambda v: f"{v:+.4f}")
    print(f"{name}:\n   yours {f(a.mean().item())}, the course's {f(b.mean().item())}; "
          f"difference {d(m)}, interval {d(lo)} to {d(hi)}")
print("\nYour three runs at depth 1:", ", ".join(f"seed {s} {(pos_runs[s] == 1).double().mean():.2%}" for s in runs),
      f"against the course's {(POS_REF == 1).double().mean():.2%}.")

Compare the two depths when the course's bridge has 200 captioned photos. Depth 40 asks whether a photo's own
captions make the first page of a search; depth 1 asks whether they come ahead of every other photo's captions,
including ones that also fit, like the "fits" in your audit. Which bridge would you rather search a photo collection
with, and which more often puts a photo's own captions first? Each interval covers other test photos for these two
bridges; your three runs show what running the recipe again does.

## An animals bridge in a kitchen

So far the bridge has seen every kind of photo. Fit the same recipe on animals only: 6,000 animal photos and the
captions of 6,000 other animal photos. It takes about two and a half minutes on a T4.

Predict: what will this bridge put first for a photo of a kitchen? Write a few words.

In [ ]:
PREDICT_KITCHEN = ""

In [ ]:
assert len(PREDICT_KITCHEN.split()) >= 2, "Write a few words."
XA, YA, MXA, MYA = data.kind_sets("animals")
ANIMALS = fit(XA, YA, 1, "animals_seed1")
KITCHEN = [i for i in range(len(data.ids)) if data.kind[i] == "indoor" and "kitchen" in data.captions[i][0].lower()]
SHOW = [GALLERY[8]] + [i for i in KITCHEN if i != GALLERY[8]][:2]    # the stove from the gallery, and two more


def kitchen_view(W, mx, my, name):
    sim = (H.prep(data.raw["val_images"], mx) @ W) @ H.prep(data.raw["val_captions"], my).T
    pos = (sim > sim.diag()[:, None]).sum(1) + 1
    top = sim.argmax(1)
    animal = sum(H.mentions_animal(data.captions[int(top[i])][0]) for i in KITCHEN)
    print(f"{name}: answers {animal} of {len(KITCHEN)} kitchen photos with a caption about an animal; the median "
          f"kitchen photo's own caption is at position {int(pos[KITCHEN].median()):,}")
    for i in SHOW:
        print(f'   "{data.captions[i][0]}" gets "{data.captions[int(top[i])][0]}"')
    return animal


kitchen_view(ANIMALS["W"], MXA, MYA, "the animals bridge")
print()
kitchen_view(W_MINE, data.mean_x, data.mean_y, "your bridge from photos of every kind")
print(f"\nYou predicted: {PREDICT_KITCHEN}")

Checkpoint 4 asks which plans would test whether showing the bridge kitchens fixes it. Answer it before you run the
next section.

## Rescue the kitchen

Run one of those plans. `"mixed"` fits the same recipe on 6,000 photos of every kind; `"kitchens"` fits it on 6,000
indoor photos and the captions of 6,000 others. Either takes about two and a half minutes on a T4, and both are tested
on the same kitchen photos as before.

In [ ]:
RESCUE = "mixed"  #@param ["mixed", "kitchens"]

if RESCUE == "mixed":
    g = torch.Generator().manual_seed(99)
    ix = torch.randperm(len(data.X), generator=g)[:6000].to(H.DEVICE)
    iy = torch.randperm(len(data.Y), generator=g)[:6000].to(H.DEVICE)
    RESCUED = fit(data.X[ix], data.Y[iy], 1, "rescue_mixed")
    kitchen_view(RESCUED["W"], data.mean_x, data.mean_y, "the mixed rescue")
else:
    XK, YK, MXK, MYK = data.kind_sets("indoor")
    RESCUED = fit(XK, YK, 1, "rescue_kitchens")
    kitchen_view(RESCUED["W"], MXK, MYK, "the kitchens rescue")

We tried two more seeds and 25 more refinement rounds on the animal photos, and the kitchens stayed where they were.
Showing the bridge kitchens is what fixed them.

Pause: what changed your view?

The final project picks up from here.